# 06 — QoS Characterisation

This notebook helps the committee visualise which workload characteristics are present in each submission, how many groups need each QoS dimension, and which submissions are most complex (multiple special requirements).

**Important:** This notebook produces visual summaries for committee use. It does **not** automatically assign queues, calculate scores, or recommend entitlement. All policy decisions remain with the committee.

**Run `00_setup.ipynb` first.**

In [ ]:
import sys
sys.path.insert(0, '/content')
sys.path.insert(0, '')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from itertools import combinations
from sheets_client import load_sheets, explode_semicolons

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

dfs = load_sheets()

submissions   = dfs.get('Submissions', pd.DataFrame())
runtime       = dfs.get('RuntimeRecords', pd.DataFrame())
mem_info      = dfs.get('MemoryInfo', pd.DataFrame())
gpu_info      = dfs.get('GpuInfo', pd.DataFrame())
ind_jobs      = dfs.get('IndependentJobs', pd.DataFrame())
pip_jobs      = dfs.get('PipelineJobs', pd.DataFrame())
ext_calcs     = dfs.get('ExtendedCalcs', pd.DataFrame())
scaling       = dfs.get('ScalingInfo', pd.DataFrame())
wtt           = dfs.get('WallTimeTerminations', pd.DataFrame())
svc           = dfs.get('ServiceObservations', pd.DataFrame())
respondents   = dfs.get('RespondentInfo', pd.DataFrame())

print(f"Submissions: {len(submissions)}")
print("Loaded all tabs.")

---
## Step 1 — Build Workload Fingerprints

For each submission a binary vector of 10 workload characteristics is derived from the data. These are observable facts derived from submitted evidence — not scores or rankings.

| Flag | Derivation |
|---|---|
| `has_long_wall_time` | Any RuntimeRecord wall_time_hours > 48 |
| `has_high_memory` | Any MemoryInfo typical_memory_gb > 128 OR peak_memory_gb > 256 |
| `has_gpu` | Any GpuInfo gpu_status = production or gpu_only |
| `has_throughput` | Has any IndependentJobs rows |
| `has_pipeline` | Has any PipelineJobs rows |
| `has_extended` | Has any ExtendedCalcs rows |
| `has_scaling` | Any ScalingInfo scaling_behaviour = linear or sublinear |
| `has_wall_time_problem` | WallTimeTerminations has_been_terminated = yes |
| `has_memory_problem` | ServiceObservations problems includes insufficient_memory |
| `has_gpu_problem` | ServiceObservations problems includes insufficient_gpu_access |

In [ ]:
def get_submission_ids(df: pd.DataFrame) -> pd.Index:
    """Return the submission_id column if present, else use the DataFrame index."""
    for col in ['submission_id', 'submissionId', 'id']:
        if col in df.columns:
            return df[col]
    return pd.Series(df.index, name='submission_id')

def get_label(df: pd.DataFrame) -> pd.Series:
    """Derive a human-readable group label from available columns."""
    for col in ['pi_name', 'group_name', 'A_pi_name', 'A_group_name']:
        if col in df.columns:
            return df[col].fillna('Unknown')
    return get_submission_ids(df).astype(str)

# Build base DataFrame from Submissions
if submissions.empty:
    print("Warning: Submissions tab is empty. Building fingerprints from submission IDs found in other tabs.")
    all_ids = set()
    for df in [runtime, mem_info, gpu_info, ind_jobs, pip_jobs, ext_calcs, scaling, wtt, svc]:
        if not df.empty:
            sid = get_submission_ids(df)
            all_ids.update(sid.tolist())
    base = pd.DataFrame({'submission_id': sorted(all_ids), 'label': sorted(all_ids)})
else:
    base = submissions.copy()
    base['submission_id'] = get_submission_ids(base)
    base['label'] = get_label(base)

base = base.set_index('submission_id')
print(f"Base index: {len(base)} submissions")

def sid_set(df: pd.DataFrame) -> set:
    """Return the set of submission IDs present in a DataFrame."""
    if df.empty:
        return set()
    return set(get_submission_ids(df).dropna().tolist())

def sid_set_where(df: pd.DataFrame, col: str, values) -> set:
    """Submission IDs in df where col is in values."""
    if df.empty or col not in df.columns:
        return set()
    mask = df[col].str.strip().str.lower().isin([v.lower() for v in values])
    return sid_set(df[mask])

def sid_set_numeric_threshold(df: pd.DataFrame, col: str, threshold: float) -> set:
    """Submission IDs where numeric col exceeds threshold."""
    if df.empty or col not in df.columns:
        return set()
    nums = df.assign(_v=pd.to_numeric(df[col], errors='coerce'))
    nums = nums[nums['_v'] > threshold]
    return sid_set(nums)

def sid_set_problems(df: pd.DataFrame, problem_key: str) -> set:
    """Submission IDs where problems_experienced contains problem_key."""
    if df.empty or 'problems_experienced' not in df.columns:
        return set()
    ids = []
    for _, row in df.iterrows():
        probs = [p.strip() for p in str(row.get('problems_experienced', '')).split(';')]
        if problem_key in probs:
            sid_col = next((c for c in ['submission_id', 'submissionId'] if c in row.index), None)
            if sid_col:
                ids.append(row[sid_col])
    return set(ids)

# Compute each fingerprint dimension
fp = pd.DataFrame(index=base.index)

fp['has_long_wall_time'] = base.index.isin(
    sid_set_numeric_threshold(runtime, 'wall_time_hours', 48)
).astype(int)

high_mem_typical = sid_set_numeric_threshold(mem_info, 'typical_memory_gb', 128)
high_mem_peak    = sid_set_numeric_threshold(mem_info, 'peak_memory_gb', 256)
fp['has_high_memory'] = base.index.isin(high_mem_typical | high_mem_peak).astype(int)

fp['has_gpu'] = base.index.isin(
    sid_set_where(gpu_info, 'gpu_status', ['production', 'gpu_only'])
).astype(int)

fp['has_throughput'] = base.index.isin(sid_set(ind_jobs)).astype(int)
fp['has_pipeline']   = base.index.isin(sid_set(pip_jobs)).astype(int)
fp['has_extended']   = base.index.isin(sid_set(ext_calcs)).astype(int)

fp['has_scaling'] = base.index.isin(
    sid_set_where(scaling, 'scaling_behaviour', ['linear', 'sublinear'])
).astype(int)

terminated_col = next((c for c in ['has_been_terminated', 'terminated'] if not wtt.empty and c in wtt.columns), None)
if terminated_col:
    fp['has_wall_time_problem'] = base.index.isin(
        sid_set_where(wtt, terminated_col, ['yes'])
    ).astype(int)
else:
    fp['has_wall_time_problem'] = 0

fp['has_memory_problem'] = base.index.isin(
    sid_set_problems(svc, 'insufficient_memory')
).astype(int)

fp['has_gpu_problem'] = base.index.isin(
    sid_set_problems(svc, 'insufficient_gpu_access')
).astype(int)

# Attach label
fp['label'] = base['label'] if 'label' in base.columns else base.index.astype(str)

print(f"Fingerprint matrix: {fp.shape}")
fp.head()

---
## Chart 2 — Workload Fingerprint Heatmap

Each row is a research group / submission. Each column is a workload characteristic. Black = characteristic present; white = absent. Rows are sorted by similarity (hierarchical clustering).

**What to look for:** Natural groupings of submissions with similar workload profiles. Groups with many black cells have complex, multi-dimensional requirements.

In [ ]:
FEATURE_COLS = [
    'has_long_wall_time', 'has_high_memory', 'has_gpu',
    'has_throughput', 'has_pipeline', 'has_extended',
    'has_scaling', 'has_wall_time_problem', 'has_memory_problem', 'has_gpu_problem'
]

FEATURE_LABELS = [
    'Long wall time\n(>48 h observed)',
    'High memory\n(>128/256 GB)',
    'GPU\n(production)',
    'Throughput\n(independent jobs)',
    'Pipeline\nworkflow',
    'Extended\ncalculation',
    'Good scaling\n(linear/sub)',
    'Wall-time\nproblem reported',
    'Memory\nproblem reported',
    'GPU access\nproblem reported',
]

fp_matrix = fp[FEATURE_COLS].copy()

if fp_matrix.empty or fp_matrix.sum().sum() == 0:
    print("Fingerprint matrix is empty — no submissions with data yet.")
else:
    # Sort rows by total number of flags (descending), then by individual flags
    fp_matrix['_total'] = fp_matrix.sum(axis=1)
    fp_matrix = fp_matrix.sort_values('_total', ascending=False).drop(columns='_total')
    fp_matrix.columns = FEATURE_LABELS

    label_series = fp.loc[fp_matrix.index, 'label'].astype(str)

    height = max(6, len(fp_matrix) * 0.35)
    fig, ax = plt.subplots(figsize=(12, height))
    sns.heatmap(
        fp_matrix,
        ax=ax,
        cmap='Blues',
        linewidths=0.4,
        linecolor='lightgrey',
        cbar=False,
        yticklabels=label_series.values,
        xticklabels=fp_matrix.columns,
        annot=False,
    )
    ax.set_title('Workload Fingerprint Heatmap\n(dark = characteristic present, sorted by total flags)', fontsize=12)
    ax.set_xlabel('')
    ax.set_ylabel('Research group / submission')
    plt.xticks(rotation=30, ha='right', fontsize=8)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.show()

---
## Chart 3 — How Many Groups Need Each QoS Dimension

Bar chart showing the count of groups where each workload characteristic is present. This is the direct evidence base for each QoS class decision.

**What to look for:** Which QoS dimensions are needed by many groups (core service) vs few groups (specialist service)? This informs capacity planning.

In [ ]:
QOS_DIMS = [
    ('Standard queue', None),  # all groups need this
    ('Long wall-time queue', 'has_long_wall_time'),
    ('High-memory nodes', 'has_high_memory'),
    ('GPU nodes', 'has_gpu'),
    ('High-throughput\n(independent jobs)', 'has_throughput'),
]

total = len(fp)
dim_labels = [d[0] for d in QOS_DIMS]
dim_counts = []
for label, col in QOS_DIMS:
    if col is None:
        dim_counts.append(total)
    elif col in fp.columns:
        dim_counts.append(int(fp[col].sum()))
    else:
        dim_counts.append(0)

fig, ax = plt.subplots(figsize=(9, 4))
colors = sns.color_palette('muted', n_colors=len(dim_labels))
bars = ax.bar(dim_labels, dim_counts, color=colors)
for bar, val in zip(bars, dim_counts):
    pct = f'{100 * val / total:.0f}%' if total > 0 else ''
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f'{val}\n({pct})', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Number of groups')
ax.set_title(f'Groups Needing Each QoS Dimension (n = {total} total)', fontsize=12)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.xticks(rotation=10, ha='right')
plt.tight_layout()
plt.show()

---
## Chart 4 — QoS Dimension Overlap (stacked bar)

Shows how many groups need combinations of the four specialist QoS dimensions (long wall time, high memory, GPU, throughput). A group needing all four is plotted in the "all four" segment.

**What to look for:** How many groups need multiple specialist dimensions? These are the most complex cases for the committee to assess.

In [ ]:
SPECIAL_DIMS = [
    ('Long wall time', 'has_long_wall_time'),
    ('High memory', 'has_high_memory'),
    ('GPU', 'has_gpu'),
    ('Throughput', 'has_throughput'),
]
SPECIAL_COLS = [c for _, c in SPECIAL_DIMS if c in fp.columns]

if not SPECIAL_COLS:
    print("No special dimension columns found in fingerprint.")
else:
    fp_sp = fp[SPECIAL_COLS].copy()
    fp_sp['n_special'] = fp_sp.sum(axis=1)

    # Count groups needing exactly 0, 1, 2, 3, 4 special dimensions
    count_by_n = fp_sp['n_special'].value_counts().sort_index()

    # For pairs — which combinations are most common?
    pair_counts = {}
    for i, j in combinations(SPECIAL_COLS, 2):
        n = int((fp_sp[i] & fp_sp[j]).sum())
        label_i = next(lbl for lbl, col in SPECIAL_DIMS if col == i)
        label_j = next(lbl for lbl, col in SPECIAL_DIMS if col == j)
        pair_counts[f'{label_i} +\n{label_j}'] = n

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: count by number of special dimensions
    ax = axes[0]
    x_labels = [f'{int(k)} special\ndimension{"s" if k != 1 else ""}' for k in count_by_n.index]
    colors = sns.color_palette('Blues_d', n_colors=len(count_by_n))
    bars = ax.bar(x_labels, count_by_n.values, color=colors)
    for bar, val in zip(bars, count_by_n.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, str(val),
                ha='center', va='bottom', fontsize=9)
    ax.set_ylabel('Number of groups')
    ax.set_title('Groups by Number of\nSpecial QoS Dimensions', fontsize=11)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    # Right: pairwise co-occurrence
    ax2 = axes[1]
    pairs = pd.Series(pair_counts).sort_values(ascending=True)
    colors2 = sns.color_palette('muted', n_colors=len(pairs))
    bars2 = ax2.barh(pairs.index, pairs.values, color=colors2)
    for bar, val in zip(bars2, pairs.values):
        ax2.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                 str(val), va='center', ha='left', fontsize=9)
    ax2.set_xlabel('Number of groups')
    ax2.set_title('Pairwise QoS Dimension Co-occurrence', fontsize=11)
    ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.suptitle('QoS Dimension Overlap', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

---
## Chart 5 / Table — Triage Order

Groups sorted by total number of special workload characteristics (most complex first). This is the natural triage order for committee review — groups at the top require the most careful assessment.

Groups with zero special dimensions likely fit the standard queue without further analysis (Category 1 triage).

Groups with many dimensions or reported problems are candidates for Category 2 (clarification needed) or Category 3 (technical assessment required).

In [ ]:
triage_cols = FEATURE_COLS  # all 10 dimensions
fp_triage = fp.copy()
fp_triage['total_flags'] = fp_triage[triage_cols].sum(axis=1)
fp_triage = fp_triage.sort_values('total_flags', ascending=False)

# Readable column names
col_rename = dict(zip(FEATURE_COLS, [
    'Long WT', 'High Mem', 'GPU',
    'Throughput', 'Pipeline', 'Extended',
    'Scaling', 'WT Problem', 'Mem Problem', 'GPU Problem'
]))

triage_display = fp_triage[['label'] + triage_cols + ['total_flags']].copy()
triage_display = triage_display.rename(columns={**col_rename, 'label': 'Group / PI', 'total_flags': 'Total flags'})

# Replace 0/1 with readable symbols
for c in col_rename.values():
    if c in triage_display.columns:
        triage_display[c] = triage_display[c].map({1: 'Y', 0: ''})

print(triage_display.to_string(index=False))

In [ ]:
# Visual version of the triage table
fp_num = fp_triage[triage_cols].copy()
fp_num['total_flags'] = fp_triage['total_flags']
labels_ordered = fp_triage['label'].astype(str).values

if len(fp_num) > 0:
    height = max(5, len(fp_num) * 0.32)
    fig, ax = plt.subplots(figsize=(13, height))

    # Show only the feature columns, not total
    display_mat = fp_num[triage_cols].values

    im = ax.imshow(display_mat, aspect='auto', cmap='Blues', vmin=0, vmax=1)

    ax.set_xticks(np.arange(len(FEATURE_LABELS)))
    ax.set_xticklabels(FEATURE_LABELS, rotation=35, ha='right', fontsize=8)
    ax.set_yticks(np.arange(len(labels_ordered)))
    ax.set_yticklabels(labels_ordered, fontsize=7)

    # Add cell text
    for i in range(len(labels_ordered)):
        for j in range(len(FEATURE_COLS)):
            val = display_mat[i, j]
            if val > 0:
                ax.text(j, i, 'Y', ha='center', va='center', fontsize=6,
                        color='white', fontweight='bold')

    # Add total flags as text on right margin
    for i, total in enumerate(fp_triage['total_flags'].values):
        ax.text(len(FEATURE_COLS) - 0.35, i, f'  {int(total)} flags',
                va='center', ha='left', fontsize=6, color='grey')

    ax.set_title(
        'Triage Order — Groups Sorted by Complexity\n'
        '(most flags at top = most complex, needs careful committee review)',
        fontsize=11
    )
    plt.tight_layout()
    plt.show()
else:
    print("No submissions to display.")

---
## Summary Statistics

In [ ]:
total_subs = len(fp)
print(f"Total submissions: {total_subs}")
print()
print("Workload characteristic prevalence:")
for col, lbl in zip(FEATURE_COLS, FEATURE_LABELS):
    n = int(fp[col].sum())
    pct = f'{100 * n / total_subs:.0f}%' if total_subs > 0 else 'n/a'
    print(f"  {lbl.replace(chr(10), ' '):40s}  {n:3d} ({pct})")

print()
fp_totals = fp[FEATURE_COLS].sum(axis=1)
print(f"Groups with 0 special characteristics (likely standard queue):  {(fp_totals == 0).sum()}")
print(f"Groups with 1 special characteristic:                           {(fp_totals == 1).sum()}")
print(f"Groups with 2+ special characteristics:                         {(fp_totals >= 2).sum()}")
print(f"Groups with 4+ special characteristics (most complex):          {(fp_totals >= 4).sum()}")